In [1]:
# Import required libraries
import pickle  
import os      
import numpy as np  
from datetime import datetime, timedelta  
import pandas as pd  
from tqdm import tqdm  
import logging  
import time  

# Setup logging configuration
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Define paths for data
DATA_PATH = "../DG_data/bluesky"
PROCESSED_DATA_PATH = "../processed_data/bluesky"

# Record start time for performance tracking
start_time = time.time()

# Load and preprocess the main dataset
logger.info("Loading data...")
df = pd.read_csv(os.path.join(DATA_PATH, 'bluesky.csv'))
# Convert unix timestamp to datetime objects
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
# Filter data from March 15, 2023 onwards
df = df[df['timestamp'] >= '2023-03-15']
# df = df[(df['timestamp'] >= '2023-03-15') & (df['timestamp'] < '2023-03-22')]

# Load user dynamic features from pickle file
with open(os.path.join(DATA_PATH, 'user_dynamic_features.pkl'), 'rb') as f:
    data = pickle.load(f)

# Increment all second-level keys by 1
user_dynamic_features = {
    outer_key: {inner_key - 1: value for inner_key, value in inner_dict.items()}
    for outer_key, inner_dict in data.items()
}

# Convert user features dictionary to DataFrame for easier manipulation
user_dynamic_features_df = pd.DataFrame.from_dict(user_dynamic_features, orient='index')
user_dynamic_features_df.index = pd.to_datetime(user_dynamic_features_df.index, unit='s')
user_dynamic_features_df = user_dynamic_features_df.sort_index()

# Add embedding date column (7am of each day) for temporal alignment
df['embedding_date'] = df['timestamp'].dt.date.apply(
    lambda x: pd.Timestamp(x) + pd.Timedelta(hours=7)
)

assert user_dynamic_features_df.columns.min() == df['source_node'].min()
assert user_dynamic_features_df.columns.max() == df['source_node'].max()
assert len(np.unique(df['source_node'])) == len(user_dynamic_features_df.columns)

# Prepare data for processing
logger.info("Preparing data...")
df = df.sort_values(['destination_node', 'timestamp'])  # Sort by post and time
grouped_posts = df.groupby('destination_node')  # Group by post ID

# Initialize list to store embedding results
all_embeddings = []

# Stats tracking
missing_user_count = 0

# Process each post's interactions
logger.info("Processing posts...")
for post_id, post_interactions in tqdm(grouped_posts): # post_id = destination_node, post_interactions = df. Basically: i,x
    """
    Example of grouped_posts structure:
    grouped_posts = {
    post_id_1: [
        {timestamp: t1, source_node: user1, ...},
        {timestamp: t2, source_node: user2, ...},
        ...
    ],
    post_id_2: [
        {timestamp: t3, source_node: user3, ...}, 
        {timestamp: t4, source_node: user4, ...},
        ...
    ],
    ...
    }
    """
    # try:
    # if post_id != 285800:
    #     continue
    # print(post_interactions.shape)

    # first_interaction = post_interactions['timestamp'].iloc[0]
    
    # Get all interactions within 24 hours of first interaction
    # end_time = first_interaction + pd.Timedelta(hours=24)
    # first_24h = post_interactions[
    #     (post_interactions['timestamp'] >= first_interaction) & 
    #     (post_interactions['timestamp'] <= end_time)
    # ]
    
    # if len(first_24h) == 0:
    #     print('this')
    #     raise ValueError()
    #     continue
        
    # Process each interaction in chronological order
    valid_embeddings = []  # Just use a list instead of pre-allocating array
    
    for i, row in post_interactions.iterrows():
        user_id = row['source_node']
        date = row['embedding_date']
        
        if user_id in user_dynamic_features_df.columns:
            embedding = user_dynamic_features_df.loc[date, user_id]
            # if isinstance(embedding, np.ndarray):
            valid_embeddings.append(embedding)
            # Calculate average of all embeddings so far
            avg_embedding = np.mean(valid_embeddings, axis=0)
            all_embeddings.append({
                'user_id': user_id,
                'post_id': post_id,
                'timestamp': row['timestamp'],
                'embedding': avg_embedding.astype(np.float16),
                'num_interactions': len(valid_embeddings)
            })
        else:
            missing_user_count += 1

    # print(len(all_embeddings))
    # raise ValueError()
                            
    # except Exception as e:
    #     logger.error(f"Error processing post {post_id}: {str(e)}")
    #     continue


# Save all embeddings directly
output_file = os.path.join(os.path.expanduser("~"), 'post_dynamic_embeddings.pkl')
output_file = '/home/sgan/post_dynamic_embeddings_v2.pkl'
with open(output_file, 'wb') as f:
    pickle.dump(all_embeddings, f)

INFO:__main__:Loading data...
INFO:__main__:Preparing data...
INFO:__main__:Processing posts...
100%|██████████| 5817132/5817132 [42:39<00:00, 2272.64it/s]  


In [2]:
len(all_embeddings)

21999631

In [ ]:
import gc
post_embeddings_df = pd.DataFrame(all_embeddings)
# del all_embeddings
gc.collect()
output_file = os.path.join(os.path.expanduser("~"), 'post_dynamic_embeddings.parquet')
post_embeddings_df.to_parquet(output_file, compression='snappy')

In [2]:
# Save all embeddings directly
output_file = os.path.join(os.path.expanduser("~"), 'post_dynamic_embeddings.pkl')
output_file = '/home/sgan/post_dynamic_embeddings_v2.pkl'
with open(output_file, 'wb') as f:
    pickle.dump(all_embeddings, f)

In [6]:
import logging
import os
import pickle

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
output_file = os.path.join(os.path.expanduser("~"), 'post_dynamic_embeddings.pkl')

# Load the saved embeddings to verify
logger.info("Loading embeddings to verify...")
with open(output_file, 'rb') as f:
    loaded_embeddings = pickle.load(f)
logger.info(f"Successfully loaded {len(loaded_embeddings)} post embeddings")

# # Create a dictionary for faster lookup by post_id and timestamp
# post_embeddings_dict = {}
# for item in loaded_embeddings:
#     post_id = item['post_id']
#     timestamp = item['timestamp']
#     if post_id not in post_embeddings_dict:
#         post_embeddings_dict[post_id] = {}
#     post_embeddings_dict[post_id][timestamp] = item['embedding']

# logger.info(f"Created lookup dictionary with {len(post_embeddings_dict)} posts")

INFO:__main__:Loading embeddings to verify...
INFO:numexpr.utils:Note: NumExpr detected 32 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
INFO:numexpr.utils:NumExpr defaulting to 8 threads.
INFO:__main__:Successfully loaded 20944583 post embeddings


In [ ]:
del loaded_embeddings